<a href="https://colab.research.google.com/github/2anoFIAP/AED-KPOP-CP-DATASCIENCE/blob/main/AEDTeste.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:

# =====================================================
# BLOCO 1 — CONFIGURAÇÃO DO AMBIENTE
# =====================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

# Configuração dos gráficos
sns.set_theme(style="whitegrid", context="notebook")

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 110

# Configurações de exibição do pandas
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

SEED = 42

print("Ambiente configurado com sucesso!")
print("Pandas:", pd.__version__)
print("Seaborn:", sns.__version__)


Ambiente configurado com sucesso!
Pandas: 2.2.3
Seaborn: 0.13.2


In [5]:

# =====================================================
# BLOCO 2 — GERAÇÃO DO DATASET SINTÉTICO
# =====================================================

rng = np.random.default_rng(SEED)

linhas = []

frete_base = {
    "Sudeste": 12,
    "Sul": 17,
    "Nordeste": 29,
    "Centro-Oeste": 22,
    "Norte": 38
}

id_pedido = 1

# Simular 4.000 clientes
for cliente_id in range(1, 4001):

    regiao = str(rng.choice(list(frete_base.keys())))

    # Perfis de utilização de cupons
    perfil_cupom = float(rng.choice([0.1, 0.5, 0.9]))

    primeira_compra = (
        pd.Timestamp("2025-01-01")
        + pd.Timedelta(days=int(rng.integers(0, 211)))
    )

    # Pelo menos uma compra inicial
    dias = [0]

    if rng.random() < 0.65:
        dias.append(int(rng.integers(1, 30)))

    # Simular recompras nos 90 dias seguintes
    media_recompras = (
        1.5 if perfil_cupom < 0.9 else 0.9
    )

    qtd_futura = int(rng.poisson(media_recompras))

    dias.extend(
        rng.integers(30, 120, size=qtd_futura).tolist()
    )

    for dia in dias:

        modalidade = str(rng.choice(
            ["ultra", "rapida", "normal"],
            p=[0.30, 0.40, 0.30]
        ))

        prazo = {
            "ultra": int(rng.integers(1, 3)),
            "rapida": int(rng.integers(3, 6)),
            "normal": int(rng.integers(6, 10))
        }[modalidade]

        receita = round(
            float(rng.gamma(4, 60) + 30), 2
        )

        cupom = bool(rng.random() < perfil_cupom)

        desconto = round(
            receita * float(rng.uniform(0.08, 0.25))
            if cupom else 0,
            2
        )

        custo_produto = round(
            receita * float(rng.uniform(0.47, 0.68)),
            2
        )

        adicional_logistico = {
            "ultra": 20,
            "rapida": 9,
            "normal": 0
        }[modalidade]

        custo_logistico = round(
            frete_base[regiao]
            + adicional_logistico
            + float(rng.uniform(0, 8)),
            2
        )

        nota_base = {
            "ultra": 4.6,
            "rapida": 4.5,
            "normal": 3.95
        }[modalidade]

        avaliacao = round(
            float(
                np.clip(
                    rng.normal(nota_base, 0.5),
                    1, 5
                )
            ),
            1
        )

        linhas.append({
            "pedido_id": id_pedido,
            "cliente_id": cliente_id,
            "data_pedido": (
                primeira_compra
                + pd.Timedelta(days=int(dia))
            ),
            "regiao": regiao,
            "receita_bruta": receita,
            "desconto_valor": desconto,
            "custo_produto": custo_produto,
            "custo_logistico": custo_logistico,
            "prazo_entrega_dias": prazo,
            "avaliacao_cliente": avaliacao,
            "cupom_aplicado": cupom
        })

        id_pedido += 1

# Criar DataFrame
df = pd.DataFrame(linhas)

# Introduzir dados ausentes de forma controlada
df["cupom_aplicado"] = df["cupom_aplicado"].astype(object)

ausencias = {
    "avaliacao_cliente": 0.04,
    "custo_logistico": 0.01,
    "cupom_aplicado": 0.01,
    "regiao": 0.005
}

for coluna, proporcao in ausencias.items():
    indices = rng.choice(
        df.index,
        size=int(len(df) * proporcao),
        replace=False
    )

    df.loc[indices, coluna] = np.nan

# Data de encerramento da observação
DATA_CORTE = pd.Timestamp("2025-12-31")

# Salvar arquivo CSV
df.to_csv("ecommerce_sintetico.csv", index=False)

print("=" * 55)
print("DATASET SINTÉTICO CRIADO")
print("=" * 55)

print("Total de pedidos:", len(df))
print("Total de clientes:", df["cliente_id"].nunique())
print("Total de regiões:", df["regiao"].nunique())
print("Data inicial:", df["data_pedido"].min())
print("Data final dos pedidos:", df["data_pedido"].max())
print("Data de corte:", DATA_CORTE.date())

display(df.head(10))


DATASET SINTÉTICO CRIADO
Total de pedidos: 11767
Total de clientes: 4000
Total de regiões: 5
Data inicial: 2025-01-01 00:00:00
Data final dos pedidos: 2025-11-24 00:00:00
Data de corte: 2025-12-31


,pedido_id,cliente_id,data_pedido,regiao,receita_bruta,desconto_valor,custo_produto,custo_logistico,prazo_entrega_dias,avaliacao_cliente,cupom_aplicado
0,1,1,2025-05-19,Sudeste,248.08,58.93,150.14,18.58,8,4.20,True
1,2,1,2025-07-27,Sudeste,157.20,32.84,85.59,39.77,2,4.50,True
2,3,2,2025-06-10,Centro-Oeste,206.53,33.01,105.29,43.04,2,4.20,True
3,4,2,2025-06-16,Centro-Oeste,346.92,46.18,223.69,48.44,2,4.70,True
4,5,3,2025-02-20,Norte,121.46,0.00,71.59,59.12,2,5.00,False
5,6,3,2025-03-12,Norte,107.76,0.00,63.17,51.47,4,4.90,False
6,7,4,2025-03-15,Centro-Oeste,204.47,0.00,113.55,48.51,2,4.30,False
7,8,4,2025-03-28,Centro-Oeste,139.22,0.00,80.08,43.22,2,4.80,False
8,9,4,2025-06-26,Centro-Oeste,240.97,24.11,161.93,38.27,4,4.00,True
9,10,4,2025-04-19,Centro-Oeste,211.55,0.00,111.52,42.77,1,4.40,False


In [6]:

# =====================================================
# BLOCO 3 — LIMPEZA E ENGENHARIA DE ATRIBUTOS
# =====================================================

# 1. Validar a unidade de análise: um pedido por linha

if df["pedido_id"].isna().any():
    raise ValueError("Existem pedidos sem ID.")

if df["pedido_id"].duplicated().any():
    raise ValueError("Existem pedidos duplicados.")

# 2. Converter colunas para os tipos adequados

colunas_numericas = [
    "receita_bruta",
    "desconto_valor",
    "custo_produto",
    "custo_logistico",
    "prazo_entrega_dias",
    "avaliacao_cliente"
]

for coluna in colunas_numericas:
    df[coluna] = pd.to_numeric(
        df[coluna],
        errors="coerce"
    )

df["data_pedido"] = pd.to_datetime(
    df["data_pedido"], errors="coerce"
)

df["regiao"] = (
    df["regiao"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

mapa_cupom = {
    "true": True,
    "false": False,
    "1": True,
    "0": False,
    "sim": True,
    "nao": False,
    "não": False,
    "yes": True,
    "no": False
}

df["cupom_aplicado"] = (
    df["cupom_aplicado"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(mapa_cupom)
    .astype("boolean")
)

# 3. Auditoria de valores ausentes

qualidade = pd.DataFrame({
    "ausentes": df.isna().sum(),
    "percentual": df.isna().mean() * 100
}).sort_values("percentual", ascending=False)

print("AUDITORIA DE VALORES AUSENTES")
display(qualidade)

# 4. Validar dados financeiros

financeiras = [
    "receita_bruta",
    "desconto_valor",
    "custo_produto",
    "custo_logistico"
]

df["financeiro_ok"] = (
    df[financeiras].notna().all(axis=1)
    & df["receita_bruta"].gt(0)
    & df["desconto_valor"].between(
        0, df["receita_bruta"]
    )
    & df["custo_produto"].ge(0)
    & df["custo_logistico"].ge(0)
)

# 5. Receita líquida

df["receita_liquida"] = (
    df["receita_bruta"] - df["desconto_valor"]
).where(df["financeiro_ok"])

# 6. Contribuição antes da logística

df["contrib_antes_logistica"] = (
    df["receita_liquida"] -
    df["custo_produto"]
).where(df["financeiro_ok"])

# 7. Contribuição efetiva por pedido

df["contribuicao"] = (
    df["contrib_antes_logistica"] -
    df["custo_logistico"]
).where(df["financeiro_ok"])

# 8. Peso logístico na receita líquida

df["taxa_logistica"] = (
    df["custo_logistico"] /
    df["receita_liquida"].replace(0, np.nan)
).where(df["financeiro_ok"])

# 9. Avaliações válidas

df["nota_valida"] = (
    df["avaliacao_cliente"]
    .where(df["avaliacao_cliente"].between(1, 5))
)

# 10. Prazos válidos

df["prazo_valido"] = (
    df["prazo_entrega_dias"]
    .where(df["prazo_entrega_dias"].ge(0))
)

# 11. Resumo da qualidade

print("\nRESUMO DA QUALIDADE")
print("Pedidos totais:", len(df))

print(
    "Pedidos financeiros válidos:",
    int(df["financeiro_ok"].sum())
)

print(
    "Pedidos financeiros inválidos/incompletos:",
    int((~df["financeiro_ok"]).sum())
)

print(
    "Avaliações válidas:",
    int(df["nota_valida"].count())
)

# 12. Margem global ponderada

base_financeira = df[df["financeiro_ok"]]

receita_total = base_financeira["receita_liquida"].sum()
contrib_total = base_financeira["contribuicao"].sum()

margem_global = (
    100 * contrib_total / receita_total
    if receita_total > 0 else np.nan
)

print(
    f"Margem global de contribuição: {margem_global:.2f}%"
)

display(df.head())


AUDITORIA DE VALORES AUSENTES


,ausentes,percentual
avaliacao_cliente,470,3.99
custo_logistico,117,0.99
cupom_aplicado,117,0.99
regiao,58,0.49
data_pedido,0,0.00
pedido_id,0,0.00
cliente_id,0,0.00
custo_produto,0,0.00
desconto_valor,0,0.00
receita_bruta,0,0.00



RESUMO DA QUALIDADE
Pedidos totais: 11767
Pedidos financeiros válidos: 11650
Pedidos financeiros inválidos/incompletos: 117
Avaliações válidas: 11297
Margem global de contribuição: 22.69%


,pedido_id,cliente_id,data_pedido,regiao,receita_bruta,desconto_valor,custo_produto,custo_logistico,prazo_entrega_dias,avaliacao_cliente,cupom_aplicado,financeiro_ok,receita_liquida,contrib_antes_logistica,contribuicao,taxa_logistica,nota_valida,prazo_valido
0,1,1,2025-05-19,Sudeste,248.08,58.93,150.14,18.58,8,4.20,True,True,189.15,39.01,20.43,0.10,4.20,8
1,2,1,2025-07-27,Sudeste,157.20,32.84,85.59,39.77,2,4.50,True,True,124.36,38.77,-1.00,0.32,4.50,2
2,3,2,2025-06-10,Centro-Oeste,206.53,33.01,105.29,43.04,2,4.20,True,True,173.52,68.23,25.19,0.25,4.20,2
3,4,2,2025-06-16,Centro-Oeste,346.92,46.18,223.69,48.44,2,4.70,True,True,300.74,77.05,28.61,0.16,4.70,2
4,5,3,2025-02-20,Norte,121.46,0.00,71.59,59.12,2,5.00,False,True,121.46,49.87,-9.25,0.49,5.00,2
